# 04 — Rebalancing Optimization

This notebook builds the station KNN edge graph, runs the minimum-cost network flow
rebalancing pipeline for each consecutive day pair in the test set, and evaluates
the resulting `s_target` assignments using the same coverage and conditional
efficiency metrics as the modeling stage. Results are exported as CSV files.

## Assumptions & Dependencies

- `02_feature_engineering.ipynb` must have written `data/processed/test.parquet`.
- `03_modeling.ipynb` must have produced a fitted LightGBM model (re-train here
  if not persisted to disk).
- Required packages: `networkx`, `lightgbm`, `pandas`, `numpy`, `matplotlib`.
- Station geometry (latitude_start, longitude_start) must be present in the test
  parquet file.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.models import train_lgbm, evaluate_coverage, coverage_summary
from src.rebalancing import build_knn_edges, adjust_to_fixed_fleet_int, run_rebalancing_pipeline

train_df = pd.read_parquet('../data/processed/train.parquet')
test_df  = pd.read_parquet('../data/processed/test.parquet')

print("Train:", train_df.shape, "| Test:", test_df.shape)

## Re-train LightGBM Model

Re-train the model on the training set so this notebook is self-contained and
does not depend on notebook 03 having persisted the model object.

In [ ]:
FEATURES = [
    'min_start_inventory_prev', 'max_start_inventory_prev',
    'station_capacity_day_prev', 'temperature_prev', 'events_prev',
    'trips_departed_prev', 'trips_arrived_prev',
    'trips_departed_roll7', 'trips_arrived_roll7',
    'temperature_roll7', 'min_start_inventory_roll7',
    'max_start_inventory_roll7', 'station_id',
]
CATEGORICAL_FEATURES = ['station_id', 'events_prev']

model = train_lgbm(train_df[FEATURES], train_df['s_true'], CATEGORICAL_FEATURES)
test_df['s_hat'] = model.predict(test_df[FEATURES])

## Station Geometry and KNN Edge Graph

Each station is connected to its k=8 nearest neighbors by Haversine distance
(in meters) using `latitude_start` and `longitude_start`. This graph defines the
set of feasible bike transfers in the min-cost flow network.

In [ ]:
stations_df = (
    test_df[['station_id', 'latitude_start', 'longitude_start']]
    .drop_duplicates(subset='station_id')
    .reset_index(drop=True)
)
edges = build_knn_edges(stations_df, k=8)
print(f"Stations: {len(stations_df)} | Edges: {len(edges)}")

## Fixed Fleet Adjustment

The total number of bikes is held constant. `s_hat` predictions are clipped to
[0, station_capacity_day] and then randomly adjusted by single bikes until they
sum to the fleet size of the first test day.

In [ ]:
first_test_day = test_df['trip_date'].min()
first_day_df = test_df[test_df['trip_date'] == first_test_day]
fleet_size = int(first_day_df['s_true'].round().clip(0).sum())
print(f"Fleet size (first test day): {fleet_size}")

## Min-Cost Flow Rebalancing Pipeline

For each consecutive pair of days in the test set, node supply/demand is computed
as `s_target_tomorrow - s_target_today`. The min-cost flow solver (NetworkX) finds
the cheapest set of transfers across the KNN graph that satisfies all supply/demand
constraints, where cost is Haversine distance in meters.

In [ ]:
test_df = run_rebalancing_pipeline(test_df, model, fleet_size, k=8)

## Post-OR KPI Evaluation

We apply the same coverage and conditional efficiency metrics as notebook 03, but
using `s_target` (the rebalanced target) instead of `s_hat_r`. This shows whether
the rebalancing step improves or degrades feasibility.

In [ ]:
test_df = evaluate_coverage(test_df, pred_col='s_target')
print(coverage_summary(test_df))

## Export Results

In [ ]:
test_df.to_csv('../reports/rebalancing_results.csv', index=False)
print("Saved rebalancing_results.csv to reports/")

## Summary

**Outputs produced by this notebook:**
- `reports/rebalancing_results.csv` — full test DataFrame with s_target, covered,
  and efficiency columns

**What the next notebook expects:** This is the final stage in the pipeline. No
further notebooks depend on these outputs.